# 06 - SQL Business Analysis & Analytical Engineering
## E-commerce Sales Analysis & Business Intelligence Dashboard
**Dataset:** UCI Online Retail II (Cleaned Sales, Returns, Adjustments & RFM Segments)  
**Tools:** SQLite 3, Python, Pandas  

---

### Objectives & Strategic Scope
This notebook translates business intelligence requirements into structured, high-performance SQL queries using an embedded SQLite database engine. It bridges raw analytical modeling and downstream enterprise data warehousing / Power BI semantic modeling.

#### Core Analytical Engineering Tasks:
1. **Database & Schema Ingestion:** Design an optimized relational SQLite database (`ecommerce_analytics.db`) housing fact and dimension tables with dedicated B-tree indexes.
2. **Basic Business Aggregations:** Validate enterprise commercial KPIs (Gross Revenue, Orders, Customers, SKUs, and AOV).
3. **Intermediate SQL Querying:** Demonstrate `GROUP BY`, `HAVING`, multi-table joins (`INNER JOIN`, `LEFT JOIN`), subqueries, `CASE WHEN` conditional categorization, and Common Table Expressions (CTEs).
4. **Advanced Analytical Window Functions:** Utilize `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `LAG()`, `LEAD()`, and `PARTITION BY` to calculate Month-over-Month (MoM) growth rates, cumulative running totals, and localized product rankings.
5. **RFM Customer Segmentation Integration:** Join commercial transaction lines with RFM segment scores to measure segment-level revenue contributions.
6. **Returns & Net Sales Reconciliation:** Isolate customer returns and calculate gross-to-net revenue impact across countries and products.
7. **Production SQL Views:** Build reusable, BI-ready database views (`vw_monthly_sales`, `vw_country_performance`, `vw_top_products`, `vw_customer_segments`, `vw_monthly_returns`).
8. **Reconciliation & Quality Validation:** Mathematically verify that SQL query results match Python EDA benchmarks with 100% accuracy.

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Paths configuration
data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
report_dir = '../reports/sql_analysis' if os.path.exists('../reports') else 'reports/sql_analysis'
sql_scripts_dir = '../sql' if os.path.exists('../sql') else 'sql'

os.makedirs(report_dir, exist_ok=True)
os.makedirs(sql_scripts_dir, exist_ok=True)

db_path = os.path.join(data_dir, 'ecommerce_analytics.db')
print(f"Target SQLite database path: {db_path}")
print(f"SQL report output directory: {report_dir}")

Target SQLite database path: ../data/processed\ecommerce_analytics.db
SQL report output directory: ../reports/sql_analysis


## 1. Database Architecture & Table Ingestion

We construct the SQLite relational database `ecommerce_analytics.db` and load all cleaned datasets:
- **`fact_sales`:** $1,007,983$ clean regular commercial sales lines.
- **`fact_returns`:** $19,104$ customer credit-note and cancellation lines.
- **`inventory_adjustments`:** $5,944$ internal warehouse stock write-downs and corrections.
- **`accounting_adjustments`:** $5$ corporate bad debt write-downs.
- **`dim_customer_rfm`:** $5,881$ registered customer accounts with RFM scores and segment labels.

We establish explicit B-tree indexes on `Invoice`, `StockCode`, `Customer_ID`, `InvoiceDate`, and `Country` to ensure sub-second analytical query performance across 1+ million rows.

In [2]:
# Connect to SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("Loading CSV datasets into SQLite tables...")

# 1. Fact Sales
sales_csv = os.path.join(data_dir, 'clean_regular_sales.csv')
df_sales = pd.read_csv(sales_csv, dtype={'Invoice': str, 'StockCode': str, 'Country': str}, low_memory=False)
df_sales['Customer_ID'] = df_sales['Customer ID'].fillna(-1).astype(int).astype(str).replace('-1', None)
df_sales.to_sql('fact_sales', conn, if_exists='replace', index=False, chunksize=50000)
print(f"[OK] Loaded 'fact_sales':             {len(df_sales):,} rows")

# 2. Fact Returns
returns_csv = os.path.join(data_dir, 'returns_and_cancellations.csv')
df_returns = pd.read_csv(returns_csv, dtype={'Invoice': str, 'StockCode': str, 'Country': str}, low_memory=False)
df_returns['Customer_ID'] = df_returns['Customer ID'].fillna(-1).astype(int).astype(str).replace('-1', None)
df_returns.to_sql('fact_returns', conn, if_exists='replace', index=False)
print(f"[OK] Loaded 'fact_returns':           {len(df_returns):,} rows")

# 3. Inventory Adjustments
inv_csv = os.path.join(data_dir, 'inventory_adjustments.csv')
df_inv = pd.read_csv(inv_csv, dtype={'Invoice': str, 'StockCode': str, 'Country': str}, low_memory=False)
df_inv.to_sql('inventory_adjustments', conn, if_exists='replace', index=False)
print(f"[OK] Loaded 'inventory_adjustments':   {len(df_inv):,} rows")

# 4. Accounting Adjustments
acc_csv = os.path.join(data_dir, 'accounting_adjustments.csv')
df_acc = pd.read_csv(acc_csv, dtype={'Invoice': str, 'StockCode': str, 'Country': str}, low_memory=False)
df_acc.to_sql('accounting_adjustments', conn, if_exists='replace', index=False)
print(f"[OK] Loaded 'accounting_adjustments':  {len(df_acc):,} rows")

# 5. Customer RFM Dimensions
rfm_csv = os.path.join(data_dir, 'rfm_customer_segments.csv')
if not os.path.exists(rfm_csv):
    rfm_csv = os.path.join('../reports/customer_analysis', 'rfm_customer_segments.csv')
df_rfm = pd.read_csv(rfm_csv, dtype={'Customer_ID': str, 'Country': str})
df_rfm.to_sql('dim_customer_rfm', conn, if_exists='replace', index=False)
print(f"[OK] Loaded 'dim_customer_rfm':        {len(df_rfm):,} rows")

# Create performance indexes
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_invoice ON fact_sales(Invoice);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_stockcode ON fact_sales(StockCode);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_customer ON fact_sales(Customer_ID);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_date ON fact_sales(InvoiceDate);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_country ON fact_sales(Country);")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_returns_customer ON fact_returns(Customer_ID);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_returns_stockcode ON fact_returns(StockCode);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_rfm_customer ON dim_customer_rfm(Customer_ID);")
conn.commit()

# Verify Table Row Counts
verification_sql = '''
SELECT 'fact_sales' AS table_name, count(*) AS total_records FROM fact_sales
UNION ALL
SELECT 'fact_returns', count(*) FROM fact_returns
UNION ALL
SELECT 'inventory_adjustments', count(*) FROM inventory_adjustments
UNION ALL
SELECT 'accounting_adjustments', count(*) FROM accounting_adjustments
UNION ALL
SELECT 'dim_customer_rfm', count(*) FROM dim_customer_rfm;
'''
df_verification = pd.read_sql_query(verification_sql, conn)
print("\n--- Database Row Count Verification ---")
display(df_verification)

Loading CSV datasets into SQLite tables...


[OK] Loaded 'fact_sales':             1,007,983 rows
[OK] Loaded 'fact_returns':           19,104 rows
[OK] Loaded 'inventory_adjustments':   5,944 rows
[OK] Loaded 'accounting_adjustments':  5 rows


[OK] Loaded 'dim_customer_rfm':        5,881 rows



--- Database Row Count Verification ---


,table_name,total_records
0,fact_sales,1007983
1,fact_returns,19104
2,inventory_adjustments,5944
3,accounting_adjustments,5
4,dim_customer_rfm,5881


## 2. Basic SQL Analysis

We execute foundational SQL queries to extract primary business performance metrics:
- Overall commercial KPIs (Revenue, Orders, Customers, SKUs, Units, AOV).
- Geographic revenue distribution.
- Monthly sales progression.
- Top 10 products by gross sales.
- Top 10 customer accounts by lifetime spend.

In [3]:
# Query 2.1: Overall Commercial KPIs
kpi_sql = '''
SELECT 
    ROUND(SUM(Revenue), 2) AS total_gross_revenue,
    COUNT(DISTINCT Invoice) AS total_orders,
    COUNT(DISTINCT Customer_ID) AS total_registered_customers,
    COUNT(DISTINCT StockCode) AS total_products_sold,
    SUM(Quantity) AS total_units_sold,
    ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS average_order_value
FROM fact_sales;
'''
print("--- Query 2.1: High-Level Commercial KPIs ---")
df_kpi_sql = pd.read_sql_query(kpi_sql, conn)
display(df_kpi_sql)

# Query 2.2: Revenue and Volume by Country (Top 10)
country_sql = '''
SELECT 
    Country,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    COUNT(DISTINCT Invoice) AS total_orders,
    COUNT(DISTINCT Customer_ID) AS total_customers,
    SUM(Quantity) AS total_units_sold,
    ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS aov,
    ROUND((SUM(Revenue) * 100.0) / (SELECT SUM(Revenue) FROM fact_sales), 2) AS revenue_share_pct
FROM fact_sales
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 10;
'''
print("\n--- Query 2.2: Top 10 Geographic Markets by Revenue ---")
df_country_sql = pd.read_sql_query(country_sql, conn)
display(df_country_sql)
df_country_sql.to_csv(os.path.join(report_dir, 'country_performance.csv'), index=False)

# Query 2.3: Monthly Revenue Trajectory
monthly_sql = '''
SELECT 
    strftime('%Y-%m', InvoiceDate) AS sales_month,
    ROUND(SUM(Revenue), 2) AS monthly_revenue,
    COUNT(DISTINCT Invoice) AS total_orders,
    SUM(Quantity) AS units_sold,
    ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS aov
FROM fact_sales
GROUP BY strftime('%Y-%m', InvoiceDate)
ORDER BY sales_month;
'''
print("\n--- Query 2.3: Monthly Sales Trajectory (Sample) ---")
df_monthly_sql = pd.read_sql_query(monthly_sql, conn)
display(df_monthly_sql.head(6))
df_monthly_sql.to_csv(os.path.join(report_dir, 'monthly_sales_summary.csv'), index=False)

# Query 2.4: Top 10 Products by Gross Sales
top_products_sql = '''
SELECT 
    StockCode,
    MAX(Description) AS Description,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    SUM(Quantity) AS total_units_sold,
    COUNT(DISTINCT Invoice) AS order_count,
    ROUND(AVG(Price), 2) AS avg_unit_price
FROM fact_sales
GROUP BY StockCode
ORDER BY total_revenue DESC
LIMIT 10;
'''
print("\n--- Query 2.4: Top 10 Products by Revenue ---")
df_top_prod_sql = pd.read_sql_query(top_products_sql, conn)
display(df_top_prod_sql)
df_top_prod_sql.to_csv(os.path.join(report_dir, 'top_products_performance.csv'), index=False)

--- Query 2.1: High-Level Commercial KPIs ---


,total_gross_revenue,total_orders,total_registered_customers,total_products_sold,total_units_sold,average_order_value
0,20476260.45,40083,5881,4917,11219901,510.85



--- Query 2.2: Top 10 Geographic Markets by Revenue ---


,Country,total_revenue,total_orders,total_customers,total_units_sold,aov,revenue_share_pct
0,United Kingdom,17410196.12,36540,5353,9201263,476.47,85.03
1,EIRE,658767.31,626,5,336579,1052.34,3.22
2,Netherlands,554038.09,229,22,384519,2419.38,2.71
3,Germany,425019.71,789,107,225173,538.68,2.08
4,France,350456.09,622,95,271902,563.43,1.71
5,Australia,169283.46,95,15,104067,1781.93,0.83
6,Spain,108332.49,154,41,50318,703.46,0.53
7,Switzerland,100685.59,93,22,52763,1082.64,0.49
8,Sweden,91869.82,105,19,88633,874.95,0.45
9,Denmark,68580.69,43,12,237471,1594.90,0.33



--- Query 2.3: Monthly Sales Trajectory (Sample) ---


,sales_month,monthly_revenue,total_orders,units_sold,aov
0,2009-12,822483.95,1682,425509,488.99
1,2010-01,651155.11,1105,390718,589.28
2,2010-02,551504.73,1203,381889,458.44
3,2010-03,830915.26,1681,526027,494.30
4,2010-04,678875.25,1462,366722,464.35
5,2010-05,657705.50,1500,395868,438.47



--- Query 2.4: Top 10 Products by Revenue ---


,StockCode,Description,total_revenue,total_units_sold,order_count,avg_unit_price
0,M,Manual,339226.24,9636,790,391.08
1,22423,REGENCY CAKESTAND 3 TIER,330590.32,26493,3919,14.18
2,DOT,DOTCOM POSTAGE,309854.11,1415,1415,218.98
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257724.71,94203,5365,3.08
4,85099B,RED RETROSPOT JUMBO BAG,180569.34,96757,3989,2.33
5,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,2.08
6,47566,PARTY BUNTING,148318.28,28204,2675,5.72
7,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,2807,1.86
8,POST,POSTAGE,125682.42,5363,1851,29.47
9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,2018,3.37


## 3. Intermediate SQL Analysis

Here we employ structured SQL techniques:
- **`GROUP BY` with `HAVING`:** Isolating high-frequency wholesale buyers ($> 10$ orders).
- **Subqueries & Common Table Expressions (CTEs):** Segmenting multi-step aggregations cleanly.
- **`CASE WHEN` Expressions:** Classifying products into pricing and volume tiers.
- **Complex Multi-Criteria Filters:** Identifying dormant high-value customer accounts.

In [4]:
# Query 3.1: High-Frequency Repeat Buyers (> 10 Orders) using HAVING
having_sql = '''
SELECT 
    Customer_ID,
    Country,
    COUNT(DISTINCT Invoice) AS order_count,
    ROUND(SUM(Revenue), 2) AS total_spend,
    SUM(Quantity) AS total_items,
    ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS aov
FROM fact_sales
WHERE Customer_ID IS NOT NULL
GROUP BY Customer_ID, Country
HAVING COUNT(DISTINCT Invoice) > 10
ORDER BY order_count DESC
LIMIT 10;
'''
print("--- Query 3.1: Top Repeat Buyers with Over 10 Orders ---")
df_having_sql = pd.read_sql_query(having_sql, conn)
display(df_having_sql)

# Query 3.2: High-Volume Low-Revenue Products (CASE WHEN + CTE)
high_vol_low_rev_sql = '''
WITH ProductSalesMetrics AS (
    SELECT 
        StockCode,
        MAX(Description) AS Description,
        SUM(Quantity) AS total_units,
        ROUND(SUM(Revenue), 2) AS total_revenue,
        ROUND(AVG(Price), 2) AS avg_unit_price
    FROM fact_sales
    GROUP BY StockCode
)
SELECT 
    StockCode,
    Description,
    total_units,
    total_revenue,
    avg_unit_price,
    CASE 
        WHEN avg_unit_price < 1.00 THEN 'Under £1 (Impulse Item)'
        WHEN avg_unit_price < 3.00 THEN 'Budget Core (£1 - £3)'
        ELSE 'Standard (£3+)'
    END AS price_tier
FROM ProductSalesMetrics
WHERE total_units > 20000 AND total_revenue < 25000
ORDER BY total_units DESC
LIMIT 10;
'''
print("\n--- Query 3.2: High-Volume Low-Revenue Fast Movers ---")
df_vol_rev_sql = pd.read_sql_query(high_vol_low_rev_sql, conn)
display(df_vol_rev_sql)

# Query 3.3: Dormant Accounts with High Historical Spend (Subquery / Inactivity Filter)
dormant_sql = '''
WITH CustomerActivity AS (
    SELECT 
        Customer_ID,
        MAX(Country) AS Country,
        ROUND(SUM(Revenue), 2) AS lifetime_spend,
        COUNT(DISTINCT Invoice) AS total_orders,
        MAX(InvoiceDate) AS last_order_date,
        CAST((julianday('2011-12-10') - julianday(MAX(InvoiceDate))) AS INTEGER) AS days_dormant
    FROM fact_sales
    WHERE Customer_ID IS NOT NULL
    GROUP BY Customer_ID
)
SELECT 
    Customer_ID,
    Country,
    lifetime_spend,
    total_orders,
    last_order_date,
    days_dormant
FROM CustomerActivity
WHERE days_dormant > 180 AND lifetime_spend > 3000
ORDER BY lifetime_spend DESC
LIMIT 10;
'''
print("\n--- Query 3.3: Dormant High-Value Accounts (>180 Days Dormant, >£3k Spend) ---")
df_dormant_sql = pd.read_sql_query(dormant_sql, conn)
display(df_dormant_sql)

--- Query 3.1: Top Repeat Buyers with Over 10 Orders ---


,Customer_ID,Country,order_count,total_spend,total_items,aov
0,14911,EIRE,398,291420.81,148222,732.21
1,12748,United Kingdom,337,53539.64,37111,158.87
2,17841,United Kingdom,211,68545.25,36584,324.86
3,15311,United Kingdom,208,114966.42,70149,552.72
4,13089,United Kingdom,203,113416.91,58937,558.70
5,14606,United Kingdom,192,29951.37,15217,156.00
6,14156,EIRE,156,313437.62,164325,2009.22
7,17850,United Kingdom,155,51208.87,21052,330.38
8,14646,Netherlands,152,528602.52,367833,3477.65
9,18102,United Kingdom,145,580987.04,181645,4006.81



--- Query 3.2: High-Volume Low-Revenue Fast Movers ---


,StockCode,Description,total_units,total_revenue,avg_unit_price,price_tier
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,106139,24445.61,0.28,Under £1 (Impulse Item)
1,17003,BROCADE RING PURSE,70369,14766.42,0.30,Under £1 (Impulse Item)
2,21213,PACK OF 72 SKULL CAKE CASES,40263,21390.76,0.74,Under £1 (Impulse Item)
3,22616,PACK OF 12 LONDON TISSUES,39147,12460.16,0.43,Under £1 (Impulse Item)
4,84568,GIRLS ALPHABET IRON ON PATCHES,37066,7788.67,0.37,Under £1 (Impulse Item)
5,84992,72 SWEETHEART FAIRY CAKE CASES,34797,17259.85,0.69,Under £1 (Impulse Item)
6,22952,SET 72 CAKE CASES VINTAGE CHRISTMAS,32988,19240.05,0.72,Under £1 (Impulse Item)
7,21982,PACK OF 12 SUKI TISSUES,32086,10395.88,0.45,Under £1 (Impulse Item)
8,21980,PACK OF 12 RED SPOTTY TISSUES,31337,8650.14,0.39,Under £1 (Impulse Item)
9,21975,PACK OF 60 DINOSAUR CAKE CASES,31332,15236.62,0.63,Under £1 (Impulse Item)



--- Query 3.3: Dormant High-Value Accounts (>180 Days Dormant, >£3k Spend) ---


,Customer_ID,Country,lifetime_spend,total_orders,last_order_date,days_dormant
0,12346,United Kingdom,77556.46,12,2011-01-18 10:01:00,325
1,16754,United Kingdom,65500.07,29,2010-12-02 17:38:00,372
2,13093,United Kingdom,54144.25,55,2011-03-09 12:14:00,275
3,17850,United Kingdom,51208.87,155,2010-12-02 15:27:00,372
4,15749,United Kingdom,44534.30,3,2011-04-18 13:20:00,235
5,15098,United Kingdom,39916.50,3,2011-06-10 15:33:00,182
6,13902,Denmark,34095.26,5,2010-03-17 13:09:00,632
7,12482,Sweden,23691.40,29,2010-05-12 16:51:00,576
8,14063,United Kingdom,22710.20,9,2010-09-27 16:18:00,438
9,15808,United Kingdom,17180.25,21,2011-02-06 14:33:00,306


## 4. Advanced SQL Analysis: Analytical Window Functions

We utilize advanced window functions supported in SQLite 3.25+:
- **`LAG()`:** Calculating Month-over-Month (MoM) revenue growth in absolute value and percentage.
- **`SUM() OVER (ORDER BY ...)`:** Calculating running cumulative revenue over time.
- **`SUM() OVER ()`:** Calculating each product's percentage contribution to total enterprise revenue.
- **`DENSE_RANK() OVER (PARTITION BY Country ...)`:** Ranking the top 3 best-selling products within each major country.

In [5]:
# Query 4.1: Month-over-Month (MoM) Revenue Growth & Cumulative Revenue (LAG + SUM OVER)
mom_sql = '''
WITH MonthlySummary AS (
    SELECT 
        strftime('%Y-%m', InvoiceDate) AS sales_month,
        ROUND(SUM(Revenue), 2) AS monthly_revenue,
        COUNT(DISTINCT Invoice) AS total_orders
    FROM fact_sales
    GROUP BY strftime('%Y-%m', InvoiceDate)
)
SELECT 
    sales_month,
    monthly_revenue,
    total_orders,
    LAG(monthly_revenue, 1) OVER (ORDER BY sales_month) AS prior_month_revenue,
    ROUND(monthly_revenue - LAG(monthly_revenue, 1) OVER (ORDER BY sales_month), 2) AS mom_revenue_change,
    ROUND(
        ((monthly_revenue - LAG(monthly_revenue, 1) OVER (ORDER BY sales_month)) / 
        LAG(monthly_revenue, 1) OVER (ORDER BY sales_month)) * 100.0, 
        2
    ) AS mom_growth_pct,
    ROUND(SUM(monthly_revenue) OVER (ORDER BY sales_month), 2) AS cumulative_running_revenue
FROM MonthlySummary
ORDER BY sales_month;
'''
print("--- Query 4.1: MoM Revenue Growth & Cumulative Progress ---")
df_mom_sql = pd.read_sql_query(mom_sql, conn)
display(df_mom_sql.head(10))
df_mom_sql.to_csv(os.path.join(report_dir, 'mom_revenue_growth.csv'), index=False)

# Query 4.2: Product Revenue Contribution Ratio using SUM() OVER ()
prod_share_sql = '''
WITH ProductRevenue AS (
    SELECT 
        StockCode,
        MAX(Description) AS Description,
        ROUND(SUM(Revenue), 2) AS product_revenue,
        SUM(Quantity) AS units_sold
    FROM fact_sales
    GROUP BY StockCode
)
SELECT 
    StockCode,
    Description,
    product_revenue,
    units_sold,
    ROUND((product_revenue * 100.0) / SUM(product_revenue) OVER (), 3) AS revenue_contribution_pct,
    ROUND(SUM(product_revenue) OVER (ORDER BY product_revenue DESC) * 100.0 / SUM(product_revenue) OVER (), 2) AS cumulative_revenue_pct,
    DENSE_RANK() OVER (ORDER BY product_revenue DESC) AS global_revenue_rank
FROM ProductRevenue
ORDER BY product_revenue DESC
LIMIT 15;
'''
print("\n--- Query 4.2: Top Products Share of Total Revenue (Window Ratio) ---")
df_prod_share_sql = pd.read_sql_query(prod_share_sql, conn)
display(df_prod_share_sql)

# Query 4.3: Top 3 Best-Selling Products per Country (DENSE_RANK OVER PARTITION BY Country)
top3_country_sql = '''
WITH CountryProductSales AS (
    SELECT 
        Country,
        StockCode,
        MAX(Description) AS Description,
        ROUND(SUM(Revenue), 2) AS product_revenue,
        SUM(Quantity) AS units_sold
    FROM fact_sales
    GROUP BY Country, StockCode
),
RankedProducts AS (
    SELECT 
        Country,
        StockCode,
        Description,
        product_revenue,
        units_sold,
        DENSE_RANK() OVER (PARTITION BY Country ORDER BY product_revenue DESC) AS rank_in_country
    FROM CountryProductSales
)
SELECT 
    Country,
    rank_in_country,
    StockCode,
    Description,
    product_revenue,
    units_sold
FROM RankedProducts
WHERE rank_in_country <= 3 AND Country IN ('United Kingdom', 'EIRE', 'Netherlands', 'Germany', 'France', 'Australia')
ORDER BY Country, rank_in_country;
'''
print("\n--- Query 4.3: Top 3 Products in Major International Markets ---")
df_top3_country_sql = pd.read_sql_query(top3_country_sql, conn)
display(df_top3_country_sql)
df_top3_country_sql.to_csv(os.path.join(report_dir, 'top_products_by_country.csv'), index=False)

--- Query 4.1: MoM Revenue Growth & Cumulative Progress ---


,sales_month,monthly_revenue,total_orders,prior_month_revenue,mom_revenue_change,mom_growth_pct,cumulative_running_revenue
0,2009-12,822483.95,1682,NaN,NaN,NaN,822483.95
1,2010-01,651155.11,1105,822483.95,-171328.84,-20.83,1473639.06
2,2010-02,551504.73,1203,651155.11,-99650.38,-15.30,2025143.79
3,2010-03,830915.26,1681,551504.73,279410.53,50.66,2856059.05
4,2010-04,678875.25,1462,830915.26,-152040.01,-18.30,3534934.30
5,2010-05,657705.50,1500,678875.25,-21169.75,-3.12,4192639.80
6,2010-06,749537.31,1645,657705.50,91831.81,13.96,4942177.11
7,2010-07,648810.27,1529,749537.31,-100727.04,-13.44,5590987.38
8,2010-08,695251.91,1425,648810.27,46441.64,7.16,6286239.29
9,2010-09,921696.99,1839,695251.91,226445.08,32.57,7207936.28



--- Query 4.2: Top Products Share of Total Revenue (Window Ratio) ---


,StockCode,Description,product_revenue,units_sold,revenue_contribution_pct,cumulative_revenue_pct,global_revenue_rank
0,M,Manual,339226.24,9636,1.66,1.66,1
1,22423,REGENCY CAKESTAND 3 TIER,330590.32,26493,1.61,3.27,2
2,DOT,DOTCOM POSTAGE,309854.11,1415,1.51,4.78,3
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,257724.71,94203,1.26,6.04,4
4,85099B,RED RETROSPOT JUMBO BAG,180569.34,96757,0.88,6.92,5
5,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,0.82,7.75,6
6,47566,PARTY BUNTING,148318.28,28204,0.72,8.47,7
7,84879,ASSORTED COLOUR BIRD ORNAMENT,129324.49,80082,0.63,9.10,8
8,POST,POSTAGE,125682.42,5363,0.61,9.72,9
9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,117760.29,35084,0.57,10.29,10



--- Query 4.3: Top 3 Products in Major International Markets ---


,Country,rank_in_country,StockCode,Description,product_revenue,units_sold
0,Australia,1,23084,RABBIT NIGHT LIGHT,3375.84,1884
1,Australia,2,22423,REGENCY CAKESTAND 3 TIER,2930.70,266
2,Australia,3,21731,RED TOADSTOOL LED NIGHT LIGHT,2464.20,1668
3,EIRE,1,M,Manual,23859.78,28
4,EIRE,2,22423,REGENCY CAKESTAND 3 TIER,15497.25,1361
5,EIRE,3,C2,CARRIAGE,9855.00,199
6,France,1,POST,POSTAGE,24400.00,1304
7,France,2,M,Manual,14513.07,10
8,France,3,23084,RABBIT NIGHT LIGHT,7277.20,4024
9,Germany,1,POST,POSTAGE,38529.20,2051


## 5. RFM Customer Segmentation Integration

We execute relational joins between `fact_sales` and `dim_customer_rfm`:
- Segment revenue, order volume, customer counts, and AOV.
- Identification of high-value accounts in the **"Cannot Lose Them"** and **"Champions"** segments.
- Geographic distribution of revenue generated by Champions.

In [6]:
# Query 5.1: Commercial Performance by RFM Segment
rfm_perf_sql = '''
SELECT 
    r.Segment,
    COUNT(DISTINCT r.Customer_ID) AS customer_count,
    COUNT(DISTINCT s.Invoice) AS total_orders,
    ROUND(SUM(s.Revenue), 2) AS segment_revenue,
    ROUND(SUM(s.Revenue) / COUNT(DISTINCT r.Customer_ID), 2) AS avg_spend_per_customer,
    ROUND(SUM(s.Revenue) / COUNT(DISTINCT s.Invoice), 2) AS aov,
    ROUND((SUM(s.Revenue) * 100.0) / (SELECT SUM(Revenue) FROM fact_sales WHERE Customer_ID IS NOT NULL), 2) AS revenue_contribution_pct
FROM dim_customer_rfm r
LEFT JOIN fact_sales s ON r.Customer_ID = s.Customer_ID
GROUP BY r.Segment
ORDER BY segment_revenue DESC;
'''
print("--- Query 5.1: RFM Segment Commercial Performance ---")
df_rfm_perf = pd.read_sql_query(rfm_perf_sql, conn)
display(df_rfm_perf)
df_rfm_perf.to_csv(os.path.join(report_dir, 'rfm_segment_performance.csv'), index=False)

# Query 5.2: Top Accounts in "Cannot Lose Them" Segment (Urgent Win-Back Candidates)
cannot_lose_sql = '''
SELECT 
    r.Customer_ID,
    r.Country,
    r.Monetary AS lifetime_gross_spend,
    r.Frequency AS lifetime_orders,
    r.Recency AS days_dormant,
    r.R_Score,
    r.F_Score,
    r.M_Score
FROM dim_customer_rfm r
WHERE r.Segment = 'Cannot Lose Them'
ORDER BY r.Monetary DESC
LIMIT 10;
'''
print("\n--- Query 5.2: Top 10 Accounts in 'Cannot Lose Them' Segment ---")
df_cannot_lose = pd.read_sql_query(cannot_lose_sql, conn)
display(df_cannot_lose)

# Query 5.3: Champions Segment Geographic Distribution
champions_country_sql = '''
SELECT 
    s.Country,
    COUNT(DISTINCT s.Customer_ID) AS champion_customers,
    COUNT(DISTINCT s.Invoice) AS champion_orders,
    ROUND(SUM(s.Revenue), 2) AS champion_revenue,
    ROUND(SUM(s.Revenue) / COUNT(DISTINCT s.Customer_ID), 2) AS avg_spend_per_champion
FROM fact_sales s
INNER JOIN dim_customer_rfm r ON s.Customer_ID = r.Customer_ID
WHERE r.Segment = 'Champions'
GROUP BY s.Country
ORDER BY champion_revenue DESC
LIMIT 8;
'''
print("\n--- Query 5.3: Geographic Distribution of Champions Revenue ---")
df_champ_country = pd.read_sql_query(champions_country_sql, conn)
display(df_champ_country)

--- Query 5.1: RFM Segment Commercial Performance ---


,Segment,customer_count,total_orders,segment_revenue,avg_spend_per_customer,aov,revenue_contribution_pct
0,Champions,1291,22128,11836647.74,9168.59,534.92,68.13
1,Loyal Customers,1147,6844,2592612.65,2260.34,378.82,14.92
2,Cannot Lose Them,224,2006,984286.42,4394.14,490.67,5.67
3,At Risk,841,2445,933225.15,1109.66,381.69,5.37
4,Potential Loyalists,415,865,391761.03,944.00,452.90,2.25
5,Hibernating,1280,1516,322732.22,252.13,212.88,1.86
6,Others,516,1004,258090.88,500.18,257.06,1.49
7,New Customers,167,167,55448.17,332.02,332.02,0.32



--- Query 5.2: Top 10 Accounts in 'Cannot Lose Them' Segment ---


,Customer_ID,Country,lifetime_gross_spend,lifetime_orders,days_dormant,R_Score,F_Score,M_Score
0,12346,United Kingdom,77556.46,12,325,2,5,5
1,16754,United Kingdom,65500.07,29,372,2,5,5
2,13093,United Kingdom,54144.25,55,275,2,5,5
3,17850,United Kingdom,51208.87,155,372,2,5,5
4,13902,Denmark,34095.26,5,632,1,4,5
5,12482,Sweden,23691.40,29,576,1,5,5
6,14063,United Kingdom,22710.20,9,438,1,5,5
7,15808,United Kingdom,17180.25,21,306,2,5,5
8,17448,United Kingdom,14523.67,46,497,1,5,5
9,13952,United Kingdom,12823.52,14,217,2,5,5



--- Query 5.3: Geographic Distribution of Champions Revenue ---


,Country,champion_customers,champion_orders,champion_revenue,avg_spend_per_champion
0,United Kingdom,1169,19812,9546631.49,8166.49
1,EIRE,2,554,604858.43,302429.22
2,Netherlands,4,176,537085.06,134271.26
3,Germany,32,578,319338.90,9979.34
4,France,29,451,271007.51,9345.09
5,Australia,3,57,156059.27,52019.76
6,Spain,6,67,75237.62,12539.60
7,Sweden,3,53,55721.39,18573.80


## 6. Returns & Net Realized Sales Analysis

We analyze the returns fact table (`fact_returns`) and reconcile Gross Revenue vs. Net Revenue:
- Return volume and refund amount by country.
- High-refund product SKUs.
- Customers with highest refund exposure.
- Country-level net revenue calculation.

In [7]:
# Query 6.1: Gross vs. Return Reconciliation by Country (Top 10)
country_recon_sql = '''
WITH GrossSales AS (
    SELECT Country, ROUND(SUM(Revenue), 2) AS gross_revenue FROM fact_sales GROUP BY Country
),
RefundTotals AS (
    SELECT Country, ROUND(ABS(SUM(Revenue)), 2) AS refund_value FROM fact_returns GROUP BY Country
)
SELECT 
    g.Country,
    g.gross_revenue,
    COALESCE(r.refund_value, 0.00) AS refund_value,
    ROUND(g.gross_revenue - COALESCE(r.refund_value, 0.00), 2) AS net_revenue,
    ROUND((COALESCE(r.refund_value, 0.00) / g.gross_revenue) * 100.0, 2) AS refund_rate_pct
FROM GrossSales g
LEFT JOIN RefundTotals r ON g.Country = r.Country
ORDER BY g.gross_revenue DESC
LIMIT 10;
'''
print("--- Query 6.1: Country Net Revenue Reconciliation ---")
df_country_recon = pd.read_sql_query(country_recon_sql, conn)
display(df_country_recon)

# Query 6.2: Top Products by Refund Value
prod_returns_sql = '''
SELECT 
    r.StockCode,
    MAX(r.Description) AS Description,
    COUNT(DISTINCT r.Invoice) AS return_orders,
    ABS(SUM(r.Quantity)) AS units_returned,
    ROUND(ABS(SUM(r.Revenue)), 2) AS total_refund_amount
FROM fact_returns r
GROUP BY r.StockCode
ORDER BY total_refund_amount DESC
LIMIT 10;
'''
print("\n--- Query 6.2: Top 10 Products by Refund Amount ---")
df_prod_returns = pd.read_sql_query(prod_returns_sql, conn)
display(df_prod_returns)

# Query 6.3: Monthly Returns Summary
monthly_returns_sql = '''
SELECT 
    strftime('%Y-%m', InvoiceDate) AS return_month,
    ROUND(ABS(SUM(Revenue)), 2) AS refund_amount,
    ABS(SUM(Quantity)) AS units_returned,
    COUNT(DISTINCT Invoice) AS return_invoices
FROM fact_returns
GROUP BY strftime('%Y-%m', InvoiceDate)
ORDER BY return_month;
'''
df_monthly_returns = pd.read_sql_query(monthly_returns_sql, conn)
df_monthly_returns.to_csv(os.path.join(report_dir, 'monthly_returns_summary.csv'), index=False)
print("\n[OK] Monthly returns summary saved to reports/sql_analysis/")

--- Query 6.1: Country Net Revenue Reconciliation ---


,Country,gross_revenue,refund_value,net_revenue,refund_rate_pct
0,United Kingdom,17410196.12,1265459.64,16144736.48,7.27
1,EIRE,658767.31,48813.53,609953.78,7.41
2,Netherlands,554038.09,5707.39,548330.70,1.03
3,Germany,425019.71,13060.55,411959.16,3.07
4,France,350456.09,28722.70,321733.39,8.20
5,Australia,169283.46,2839.04,166444.42,1.68
6,Spain,108332.49,17319.05,91013.44,15.99
7,Switzerland,100685.59,1282.53,99403.06,1.27
8,Sweden,91869.82,4094.30,87775.52,4.46
9,Denmark,68580.69,4121.10,64459.59,6.01



--- Query 6.2: Top 10 Products by Refund Amount ---


,StockCode,Description,return_orders,units_returned,total_refund_amount
0,M,Manual,496,5443,422552.57
1,AMAZONFEE,AMAZON FEE,33,33,241988.30
2,23843,"PAPER CRAFT , LITTLE BIRDIE",1,80995,168469.60
3,23166,MEDIUM CERAMIC TOP STORAGE JAR,10,74494,77479.64
4,BANK CHARGES,Bank Charges,62,76,36001.49
5,22423,REGENCY CAKESTAND 3 TIER,341,1450,16545.30
6,POST,POSTAGE,226,302,15252.01
7,D,Discount,146,3064,13277.52
8,85123A,WHITE HANGING HEART T-LIGHT HOLDER,134,3637,9387.10
9,CRUK,CRUK Commission,16,16,7933.43



[OK] Monthly returns summary saved to reports/sql_analysis/


## 7. Reusable SQL Views for Power BI Integration

We deploy 5 persistent SQL views inside `ecommerce_analytics.db`:
1. **`vw_monthly_sales`:** Aggregates monthly revenue, orders, active customers, units, and AOV.
2. **`vw_country_performance`:** Consolidates geographic metrics and market share percentages.
3. **`vw_top_products`:** Products ranked by revenue with average selling prices.
4. **`vw_customer_segments`:** Segment-level commercial performance and contribution metrics.
5. **`vw_monthly_returns`:** Monthly refund totals, returned units, and return order volume.

These views provide clean semantic entry points for Power BI DirectQuery or Import models.

In [8]:
# Deploy SQL Views
cursor.execute("DROP VIEW IF EXISTS vw_monthly_sales;")
cursor.execute('''
CREATE VIEW vw_monthly_sales AS
SELECT 
    strftime('%Y-%m', InvoiceDate) AS sales_month,
    ROUND(SUM(Revenue), 2) AS gross_revenue,
    COUNT(DISTINCT Invoice) AS total_orders,
    COUNT(DISTINCT Customer_ID) AS active_registered_customers,
    SUM(Quantity) AS total_units_sold,
    ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS aov
FROM fact_sales
GROUP BY strftime('%Y-%m', InvoiceDate);
''')

cursor.execute("DROP VIEW IF EXISTS vw_country_performance;")
cursor.execute('''
CREATE VIEW vw_country_performance AS
SELECT 
    Country,
    ROUND(SUM(Revenue), 2) AS gross_revenue,
    COUNT(DISTINCT Invoice) AS total_orders,
    COUNT(DISTINCT Customer_ID) AS total_customers,
    SUM(Quantity) AS total_units_sold,
    ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS aov,
    ROUND((SUM(Revenue) * 100.0) / (SELECT SUM(Revenue) FROM fact_sales), 2) AS market_share_pct
FROM fact_sales
GROUP BY Country;
''')

cursor.execute("DROP VIEW IF EXISTS vw_top_products;")
cursor.execute('''
CREATE VIEW vw_top_products AS
SELECT 
    StockCode,
    MAX(Description) AS Description,
    ROUND(SUM(Revenue), 2) AS gross_revenue,
    SUM(Quantity) AS total_units_sold,
    COUNT(DISTINCT Invoice) AS total_orders,
    ROUND(AVG(Price), 2) AS avg_unit_price,
    DENSE_RANK() OVER (ORDER BY SUM(Revenue) DESC) AS revenue_rank
FROM fact_sales
GROUP BY StockCode;
''')

cursor.execute("DROP VIEW IF EXISTS vw_customer_segments;")
cursor.execute('''
CREATE VIEW vw_customer_segments AS
SELECT 
    r.Segment,
    COUNT(DISTINCT r.Customer_ID) AS total_customers,
    COUNT(DISTINCT s.Invoice) AS total_orders,
    ROUND(SUM(s.Revenue), 2) AS segment_revenue,
    ROUND(SUM(s.Revenue) / COUNT(DISTINCT r.Customer_ID), 2) AS avg_customer_spend,
    ROUND(SUM(s.Revenue) / COUNT(DISTINCT s.Invoice), 2) AS aov,
    ROUND((SUM(s.Revenue) * 100.0) / (SELECT SUM(Revenue) FROM fact_sales WHERE Customer_ID IS NOT NULL), 2) AS revenue_contribution_pct
FROM dim_customer_rfm r
LEFT JOIN fact_sales s ON r.Customer_ID = s.Customer_ID
GROUP BY r.Segment;
''')

cursor.execute("DROP VIEW IF EXISTS vw_monthly_returns;")
cursor.execute('''
CREATE VIEW vw_monthly_returns AS
SELECT 
    strftime('%Y-%m', InvoiceDate) AS return_month,
    ROUND(ABS(SUM(Revenue)), 2) AS total_refund_amount,
    ABS(SUM(Quantity)) AS units_returned,
    COUNT(DISTINCT Invoice) AS return_orders,
    COUNT(*) AS return_line_items
FROM fact_returns
GROUP BY strftime('%Y-%m', InvoiceDate);
''')

conn.commit()
print("[OK] All 5 SQL Views successfully compiled in SQLite database.")

# Sample view inspection
print("\n--- Sample Output from vw_monthly_sales ---")
display(pd.read_sql_query("SELECT * FROM vw_monthly_sales LIMIT 5;", conn))

[OK] All 5 SQL Views successfully compiled in SQLite database.

--- Sample Output from vw_monthly_sales ---


,sales_month,gross_revenue,total_orders,active_registered_customers,total_units_sold,aov
0,2009-12,822483.95,1682,955,425509,488.99
1,2010-01,651155.11,1105,720,390718,589.28
2,2010-02,551504.73,1203,774,381889,458.44
3,2010-03,830915.26,1681,1057,526027,494.30
4,2010-04,678875.25,1462,942,366722,464.35


## 8. Quality Validation & Cross-Tool Reconciliation

We rigorously cross-validate the SQL outputs against the prior Python EDA benchmarks:
- **Total Revenue:** £20,476,260.45 (100.00% exact match).
- **Total Registered Customers:** 5,881 accounts (100.00% exact match).
- **Total Physical Units:** 11,219,901 units (100.00% exact match).
- **Distinct Order Count:** 40,083 orders (validated against string-typed invoice keys).
- **Refund Monetary Value:** -£1,462,050.61 across 19,104 cancellation lines.

In [9]:
# Extract benchmark figures via SQL
sql_val = pd.read_sql_query('''
SELECT 
    ROUND(SUM(Revenue), 2) AS sql_revenue,
    COUNT(DISTINCT Invoice) AS sql_orders,
    COUNT(DISTINCT Customer_ID) AS sql_customers,
    SUM(Quantity) AS sql_quantity
FROM fact_sales;
''', conn).iloc[0]

# Expected benchmarks from Python Cleaning/EDA
expected_revenue = 20476260.45
expected_customers = 5881
expected_quantity = 11219901
expected_orders = 40083

print("--- SQL vs. Python Benchmark Comparison ---")
print(f"Gross Revenue:         SQL = £{sql_val['sql_revenue']:,.2f} | Expected = £{expected_revenue:,.2f} | Match: {sql_val['sql_revenue'] == expected_revenue}")
print(f"Registered Customers:  SQL = {sql_val['sql_customers']:,}      | Expected = {expected_customers:,}      | Match: {sql_val['sql_customers'] == expected_customers}")
print(f"Physical Units Sold:   SQL = {sql_val['sql_quantity']:,} | Expected = {expected_quantity:,} | Match: {sql_val['sql_quantity'] == expected_quantity}")
print(f"Distinct Invoices:     SQL = {sql_val['sql_orders']:,}     | Expected = {expected_orders:,}     | Match: {sql_val['sql_orders'] == expected_orders}")

# Programmatic assertion
assert sql_val['sql_revenue'] == expected_revenue, "Revenue discrepancy detected!"
assert sql_val['sql_customers'] == expected_customers, "Customer count discrepancy detected!"
assert sql_val['sql_quantity'] == expected_quantity, "Quantity discrepancy detected!"
assert sql_val['sql_orders'] == expected_orders, "Order count discrepancy detected!"

print("\n✓ ALL MATHEMATICAL VALIDATION CHECKS PASSED WITH 100.00% ACCURACY.")

--- SQL vs. Python Benchmark Comparison ---
Gross Revenue:         SQL = £20,476,260.45 | Expected = £20,476,260.45 | Match: True
Registered Customers:  SQL = 5,881.0      | Expected = 5,881      | Match: True
Physical Units Sold:   SQL = 11,219,901.0 | Expected = 11,219,901 | Match: True
Distinct Invoices:     SQL = 40,083.0     | Expected = 40,083     | Match: True

✓ ALL MATHEMATICAL VALIDATION CHECKS PASSED WITH 100.00% ACCURACY.


## 9. Deliverables & Data Assets Summary

### Artifacts Produced:
1. **SQLite Database File:** `data/processed/ecommerce_analytics.db` ($227\text{ MB}$) with 5 tables and 5 analytical views.
2. **Production SQL Scripts in `sql/`:**
   - [`sql/01_schema_and_views.sql`](file:///c:/Users/sabha/OneDrive/Desktop/Freelance/Ecommerce-Sales-Analytics/sql/01_schema_and_views.sql): Full DDL, performance indexes, and view definitions.
   - [`sql/02_analytical_queries.sql`](file:///c:/Users/sabha/OneDrive/Desktop/Freelance/Ecommerce-Sales-Analytics/sql/02_analytical_queries.sql): Catalog of all basic, intermediate, and window function business queries.
3. **Query Result Exports in `reports/sql_analysis/`:**
   - `monthly_sales_summary.csv`
   - `country_performance.csv`
   - `top_products_performance.csv`
   - `rfm_segment_performance.csv`
   - `monthly_returns_summary.csv`
   - `top_products_by_country.csv`
   - `mom_revenue_growth.csv`

In [10]:
conn.close()
print("Database connection closed cleanly.")

print(f"\nDeliverable Files in {report_dir}:")
for f in sorted(os.listdir(report_dir)):
    p = os.path.join(report_dir, f)
    print(f" - {f} ({os.path.getsize(p) / 1024:.2f} KB)")

print(f"\nDatabase File Status:")
print(f" - {db_path} ({os.path.getsize(db_path) / (1024*1024):.2f} MB)")

Database connection closed cleanly.

Deliverable Files in ../reports/sql_analysis:
 - country_performance.csv (0.54 KB)
 - mom_revenue_growth.csv (1.60 KB)
 - monthly_returns_summary.csv (0.73 KB)
 - monthly_sales_summary.csv (0.98 KB)
 - rfm_segment_performance.csv (0.52 KB)
 - top_products_by_country.csv (0.96 KB)
 - top_products_performance.csv (0.61 KB)

Database File Status:
 - ../data/processed\ecommerce_analytics.db (227.14 MB)
